# Topic 2: Advanced Joins — Deep Dive

> **Phase 7 → Week 12 → Topic 2**

---

## Join Algorithms in Spark

```
1. Broadcast Hash Join (BHJ):
   Condition: one table small enough to broadcast (< autoBroadcastJoinThreshold = 10 MB)
   How: small table broadcast to all executors → each executor does local hash join
   No shuffle. Fastest join.
   
   ┌─────────────┐     broadcast     ┌──────────────────┐
   │ Small Table │ ────────────────► │ All executors    │
   │ (< 10 MB)   │                   │ hold full copy   │
   └─────────────┘                   │ of small table   │
                                     └──────────────────┘
                                           ↑ local join ↑
                                     ┌──────────────────┐
                                     │ Big table chunks │
                                     └──────────────────┘

2. Sort-Merge Join (SMJ):
   Condition: both tables are large (> broadcast threshold)
   How: shuffle both tables by join key → sort each partition → merge
   Two shuffles (one per table) → expensive but handles any size
   Default for large table joins.

3. Shuffle Hash Join (SHJ):
   Like SMJ but skips the sort — builds hash table from smaller shuffled side
   Faster than SMJ when smaller side fits in memory
   Enabled by AQE when it detects the smaller partition fits in memory

4. Broadcast Nested Loop Join:
   No join key / cartesian product / non-equi join
   O(n×m) — avoid unless tables are very small
```

---

## Interview Cheat Sheet

**Q: When does Spark choose Broadcast Hash Join over Sort-Merge Join?**
> Spark uses BHJ when one table is smaller than `spark.sql.autoBroadcastJoinThreshold` (default 10 MB). The threshold compares the table's estimated size in bytes (from statistics). You can force BHJ with `F.broadcast(df)` hint even if the table is above the threshold. BHJ is always faster because it avoids the shuffle — the small table is copied to every executor.

**Q: What is a Bloom Filter join and when is it useful?**
> A Bloom Filter is a probabilistic data structure that can quickly tell if an element is definitely NOT in a set (with 0 false negatives) or might be in the set (with a small false positive rate). In Spark 3.3+, AQE can build a Bloom Filter from the join key values of the smaller table and use it to filter the larger table BEFORE the shuffle. This eliminates rows that will definitely not match, reducing shuffle data size significantly. Best for: selective joins (most rows in the larger table don't match).

**Q: What is a skewed join and how do you fix it?**
> A skewed join happens when some join keys have disproportionately many rows — e.g., key="unknown" has 80% of the data. The partition handling that key takes 100× longer than others. Fix options: (1) AQE skew join handling (automatic in Spark 3.0+ with `spark.sql.adaptive.skewJoin.enabled=true`). (2) Salting: add a random suffix to the skewed key on both sides, split into sub-partitions. (3) Separate the skewed key and process it with broadcast join.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("Week12 - Advanced Joins") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

## Part 1: Broadcast Hash Join — Force and Verify

In [ ]:
# Large orders table
orders = spark.range(100_000) \
    .withColumn("order_id",   F.concat(F.lit("O"), F.col("id").cast("string"))) \
    .withColumn("customer_id",F.concat(F.lit("C"), (F.col("id") % 100).cast("string"))) \
    .withColumn("amount",     (F.col("id") % 500 + 10).cast("double")) \
    .withColumn("category",   F.element_at(
        F.array(F.lit("electronics"), F.lit("clothing"), F.lit("food")),
        (F.col("id") % 3 + 1).cast("int")
    )).drop("id")

# Small lookup table
categories = spark.createDataFrame([
    ("electronics", 0.18, "EL"),
    ("clothing",    0.12, "CL"),
    ("food",        0.05, "FD"),
], ["category", "tax_rate", "code"])

print("=== Auto Broadcast (threshold = 10 MB) ===")
auto_join = orders.join(categories, "category")
auto_join.explain(mode="formatted")  # Look for BroadcastHashJoin in plan

In [ ]:
# Force broadcast with hint (works even above autoBroadcastJoinThreshold)
print("=== Forced Broadcast with F.broadcast() hint ===")
forced_broadcast = orders.join(F.broadcast(categories), "category")
forced_broadcast.explain(mode="formatted")

# Performance comparison
t0 = time.time()
forced_broadcast.agg(F.sum("amount")).collect()
broadcast_time = time.time() - t0

# Disable broadcast to force Sort-Merge Join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
t0 = time.time()
orders.join(categories, "category").agg(F.sum("amount")).collect()
smj_time = time.time() - t0
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

print(f"\nBroadcast Hash Join: {broadcast_time:.3f}s")
print(f"Sort-Merge Join:     {smj_time:.3f}s")
print(f"Speedup: {smj_time/broadcast_time:.1f}x")

## Part 2: Join Hints

In [ ]:
# Spark 3.x join hints — force the optimizer to use a specific algorithm
print("""
Join Hints Reference:
════════════════════════════════════════════════════════════════

# Broadcast Hash Join (small table)
df1.join(df2.hint("broadcast"), "key")
df1.hint("broadcast").join(df2, "key")

# Sort-Merge Join (large tables, both shuffled)
df1.hint("merge").join(df2, "key")

# Shuffle Hash Join (build hash from smaller side after shuffle)
df1.join(df2.hint("shuffle_hash"), "key")

# Shuffle Replicate Nested Loop (non-equi joins / cartesian)
df1.hint("shuffle_replicate_nl").crossJoin(df2)

SQL equivalents:
  SELECT /*+ BROADCAST(t2) */ * FROM t1 JOIN t2 ON t1.key = t2.key
  SELECT /*+ MERGE(t1, t2) */ * FROM t1 JOIN t2 ON t1.key = t2.key

Priority: hints > AQE > Catalyst rules > statistics-based optimizer
════════════════════════════════════════════════════════════════
""")

# Demonstrate different hints
for hint_name in ["broadcast", "merge", "shuffle_hash"]:
    print(f"=== {hint_name.upper()} hint ===")
    hinted = orders.join(categories.hint(hint_name), "category")
    plan = hinted._jdf.queryExecution().simpleString()
    # Look for the join type in the plan
    if "BroadcastHashJoin" in plan:
        print("  Plan: BroadcastHashJoin")
    elif "SortMergeJoin" in plan:
        print("  Plan: SortMergeJoin")
    elif "ShuffledHashJoin" in plan:
        print("  Plan: ShuffledHashJoin")
    else:
        print("  Plan: see explain()")

## Part 3: Skewed Join — Problem and Fix

In [ ]:
# Create a skewed dataset: category 'electronics' has 90% of rows
from pyspark.sql.functions import rand

skewed_orders = spark.range(50_000) \
    .withColumn("category",
        F.when(F.col("id") < 45_000, F.lit("electronics"))  # 90% skewed
         .when(F.col("id") < 48_000, F.lit("clothing"))      # 6%
         .otherwise(F.lit("food"))                            # 4%
    ) \
    .withColumn("amount", (F.col("id") % 200 + 1).cast("double"))

print("Skew distribution:")
skewed_orders.groupBy("category").count().orderBy("count", ascending=False).show()

# AQE automatically detects and splits skewed partitions
# Enable: spark.sql.adaptive.skewJoin.enabled=true (default in Spark 3.x)
result = skewed_orders.join(categories, "category") \
    .groupBy("category") \
    .agg(F.count("*").alias("count"), F.sum("amount").alias("total"))
result.show()

print("AQE skew join: Spark split the large 'electronics' partition automatically")
print("Check Spark UI > SQL > physical plan for 'AQEShuffleRead' nodes")

In [ ]:
# Manual fix: salting (when AQE is not available or skew is extreme)
SALT_FACTOR = 5

# Salt the large table: distribute skewed key across N buckets
salted_orders = skewed_orders \
    .withColumn("salt", (F.rand() * SALT_FACTOR).cast("int")) \
    .withColumn("salted_category", F.concat(F.col("category"), F.lit("_"), F.col("salt").cast("string")))

# Expand the small table: replicate each row N times with all salt values
salted_categories = categories \
    .withColumn("salt", F.explode(F.array([F.lit(i) for i in range(SALT_FACTOR)]))) \
    .withColumn("salted_category", F.concat(F.col("category"), F.lit("_"), F.col("salt").cast("string")))

# Join on salted key → evenly distributed
salted_result = salted_orders \
    .join(F.broadcast(salted_categories), "salted_category") \
    .drop("salt", "salted_category") \
    .groupBy(salted_orders.category) \
    .agg(F.count("*").alias("count"), F.sum("amount").alias("total"))

print("Salted join result (same answer, distributed evenly):")
salted_result.show()

## Part 4: Bloom Filter Joins (Spark 3.3+)

In [ ]:
print("""
Bloom Filter Join (AQE in Spark 3.3+):
════════════════════════════════════════════════════════════════

What it does:
  Before shuffling the large table, AQE builds a Bloom Filter from
  the join keys of the SMALLER table (after it's been scanned).
  The Bloom Filter is broadcast to all executors processing the LARGE table.
  Each row in the large table is tested against the filter:
    → "definitely not in small table": row DROPPED before shuffle
    → "might be in small table":       row included in shuffle

When it helps:
  Large table has many rows that DON'T match any join key in the small table.
  Example: 1B user events joined with 10K VIP customer IDs
    → Most events are from non-VIP users → filtered early → 99% shuffle reduction

When it doesn't help:
  High selectivity — most rows in both tables match.
  Very small tables (Broadcast Hash Join is better).

Enable:
  spark.conf.set("spark.sql.optimizer.runtime.bloomFilter.enabled", "true")
  spark.conf.set("spark.sql.optimizer.runtime.bloomFilter.creationSideThreshold", "10m")

Verify in Spark UI: look for "BloomFilter" in the Physical Plan

Manual Bloom Filter (pure Python, for illustration):
════════════════════════════════════════════════════════════════
""")

# Manual Bloom filter demo (illustrative — Spark does this automatically in 3.3+)
import math

class SimpleBloomFilter:
    def __init__(self, n, fp_rate=0.01):
        self.size = int(-n * math.log(fp_rate) / math.log(2)**2)
        self.bit_array = [False] * self.size
        self.k = int(self.size / n * math.log(2))

    def _hashes(self, item):
        return [hash(f"{item}_{i}") % self.size for i in range(self.k)]

    def add(self, item):
        for h in self._hashes(item):
            self.bit_array[h] = True

    def might_contain(self, item):
        return all(self.bit_array[h] for h in self._hashes(item))

# Small side: VIP customer IDs
vip_ids = {"C001", "C050", "C100", "C200"}
bf = SimpleBloomFilter(len(vip_ids))
for cid in vip_ids:
    bf.add(cid)

# Large side: 20 customer IDs, most not VIP
all_customers = [f"C{i:03d}" for i in range(1, 21)]
filtered = [c for c in all_customers if bf.might_contain(c)]
actual_matches = [c for c in all_customers if c in vip_ids]

print(f"All customers: {len(all_customers)}")
print(f"After Bloom filter: {len(filtered)} (may include false positives)")
print(f"Actual matches: {len(actual_matches)}")
print(f"Filtered: {filtered}")

## Exercises

1. Create two DataFrames: `orders` (1M rows) and `regions` (5 rows). Join them. Check the explain plan — is Spark using BroadcastHashJoin? Now set `autoBroadcastJoinThreshold=-1` and re-check. What changed?
2. Create a skewed dataset where `key='HOT'` has 95% of rows. Do a join without AQE (disable it). Measure wall time. Then enable AQE and repeat. What speedup do you see?
3. Implement the salting pattern manually for a skewed join. Verify the result matches the non-salted result using `assert_df_equal` (from Week 9).
4. When is a Sort-Merge Join preferred over a Broadcast Hash Join? Describe a scenario where forcing BHJ would cause OOM on the executor.
5. Explain why Bloom Filters have false positives but no false negatives. Why is this acceptable for join filtering?